# 📷 → 🧊  Reconstrucción 3D con Gaussian Splatting

Sube un **.zip con fotos** de un objeto y obtén un **modelo 3D** que puedes girar en el navegador.
Todo corre en la **GPU gratuita de Google Colab**, sin instalar nada en tu ordenador.

**Repo:** https://github.com/emr81-ua/3d-gaussian-splatting-reconstruction

---
### Cómo usarlo
1. Menú **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)**.
2. Ejecuta las celdas **de arriba a abajo** (▶ en cada una).
3. Cuando te lo pida, sube tu `.zip` de fotos.
4. Al final descargas el modelo `.ply` y lo ves online.

> ⏱️ La instalación tarda unos minutos y el entrenamiento otros tantos. Ten paciencia.


## 1. Comprobar la GPU


In [ ]:
# Debe aparecer una GPU (T4). Si no, ve a  Entorno de ejecucion -> Cambiar tipo de entorno -> GPU.
!nvidia-smi


## 2. Instalar las herramientas
COLMAP (poses de cámara) + nerfstudio (entrenamiento 3D Gaussian Splatting).


In [ ]:
# Instalacion (varios minutos la primera vez).
!sudo apt-get -qq update
!sudo apt-get -qq install -y colmap
!pip install -q nerfstudio
print('
Instalacion terminada.')
print('Ignora los avisos amarillos de pip: son de paquetes de Colab (tensorflow, jax...) que aqui no usamos.')


## 3. Sube tu .zip de fotos
30–60 fotos dando la vuelta al objeto, con **solape** entre una y la siguiente, buena luz y fondo con textura.


In [ ]:
import zipfile, os, shutil
from google.colab import files

shutil.rmtree('/content/photos', ignore_errors=True)
shutil.rmtree('/content/_unzip', ignore_errors=True)
os.makedirs('/content/photos', exist_ok=True)

print('Sube tu .zip de fotos:')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content/_unzip')

exts = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')
n = 0
for root, _, fs in os.walk('/content/_unzip'):
    for f in sorted(fs):
        if f.lower().endswith(exts):
            ext = os.path.splitext(f)[1].lower()
            shutil.copy(os.path.join(root, f), f'/content/photos/{n:04d}{ext}')
            n += 1
print(f'
{n} fotos listas en /content/photos')
assert n >= 5, 'Muy pocas fotos: sube al menos 15-20 para una reconstruccion decente.'


## 4. Estimar poses de cámara (COLMAP)
⚠️ El COLMAP de Colab es **solo CPU**, por eso usamos `--no-gpu`. Tarda unos minutos con 30–60 fotos.


In [ ]:
# limpiar intentos previos para que no se mezclen
!rm -rf /content/processed /content/outputs

!ns-process-data images \
    --data /content/photos \
    --output-dir /content/processed \
    --no-gpu \
    --verbose


### Comprobación: ¿registró bien las cámaras?
Si esto falla, el problema son las fotos (poco solape / poca textura), no el resto del notebook.


In [ ]:
import os, json
tj = '/content/processed/transforms.json'
if not os.path.exists(tj):
    raise SystemExit('COLMAP no genero transforms.json. Revisa el log de la celda 4: '
                     'probablemente las fotos tienen poco solape o poca textura.')
frames = json.load(open(tj)).get('frames', [])
print(f'transforms.json OK -- COLMAP registro {len(frames)} camaras.')
assert len(frames) >= 3, 'COLMAP registro muy pocas camaras; el resultado sera pobre. Prueba con mas fotos o mejor solape.'


## 5. Entrenar el modelo 3D Gaussian Splatting
Debe tardar **varios minutos** (si acaba en segundos, algo fue mal antes). Para una prueba rápida baja `--max-num-iterations` a 7000.


In [ ]:
!ns-train splatfacto \
    --data /content/processed \
    --max-num-iterations 15000 \
    --viewer.quit-on-train-completion True \
    --output-dir /content/outputs


## 6. Exportar el modelo a .ply


In [ ]:
import glob
configs = sorted(glob.glob('/content/outputs/**/config.yml', recursive=True))
assert configs, 'No hay config.yml: el entrenamiento (celda 5) no llego a correr.'
config = configs[-1]
print('Usando config:', config)
!ns-export gaussian-splat --load-config "$config" --output-dir /content/export


## 7. Descargar el modelo


In [ ]:
import glob, os
from google.colab import files
plys = sorted(glob.glob('/content/export/**/*.ply', recursive=True))
assert plys, 'No se genero ningun .ply en /content/export.'
modelo = plys[-1]
print('Modelo 3D:', modelo, '(', round(os.path.getsize(modelo)/1e6, 1), 'MB )')
files.download(modelo)


## 8. Verlo online
Arrastra el `.ply` descargado a cualquiera de estos visores en el navegador:

- **SuperSplat**: https://playcanvas.com/supersplat/editor
- **antimatter15 splat viewer**: https://antimatter15.com/splat/

---
### ⚠️ Notas y problemas comunes
- **COLMAP en Colab es solo CPU** (sin CUDA), por eso el `--no-gpu` de la celda 4. Es más lento pero funciona.
- Si la celda 5 **acaba en segundos**: COLMAP no registró cámaras (celda 4). Revisa el solape/textura de tus fotos.
- Si ves errores raros de `numpy`: menú **Entorno de ejecución → Reiniciar sesión**, y re-ejecuta desde la celda 3 (sin repetir la 2).
- Este Colab usa **nerfstudio (splatfacto)** para poder correr gratis en la nube. La versión de escritorio del proyecto usa **LichtFeld Studio**; el pipeline es equivalente.
